In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler,MinMaxScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.decomposition import PCA

In [3]:
df = pd.read_csv("01 covid_toy.csv")
display(df.head())

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [4]:
print(df.dtypes)

age            int64
gender           str
fever        float64
cough            str
city             str
has_covid        str
dtype: object


In [6]:
numeric_columns = df.select_dtypes(include=np.number).columns

print("Numeric Columns:")
print(numeric_columns)

Numeric Columns:
Index(['age', 'fever'], dtype='str')


In [7]:
categorical_columns = df.select_dtypes(include="object").columns

print("categorical Columns:")
print(categorical_columns)

categorical Columns:
Index(['gender', 'cough', 'city', 'has_covid'], dtype='str')


C:\Users\student\AppData\Local\Temp\ipykernel_292\3737727741.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include="object").columns


In [9]:
df["has_covid"] = df["has_covid"].astype("object")

In [10]:
print(df.isnull().sum())

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64


In [11]:
df = df.replace("?",np.nan)

In [12]:
print(df.isnull().sum())

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64


In [14]:
df["fever"]= pd.to_numeric(df["fever"])

In [15]:
print(df.isnull().sum())

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64


In [16]:
df = df.replace("?",np.nan)

In [17]:
print(df.isnull().sum())

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64


In [18]:
df["age"]= pd.to_numeric(df["age"])

In [19]:
df["age"]= df["age"].fillna(df["age"].mean())

In [20]:
print(df.isnull().sum())

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64


In [21]:
scaler =MinMaxScaler().set_output(transform="pandas")
scaled_data = scaler.fit_transform(df[numeric_columns])
scaled_data.head()

,age,fever
0,0.696203,0.833333
1,0.278481,0.333333
2,0.468354,0.500000
3,0.329114,0.000000
4,0.759494,0.500000


In [22]:
categorical_columns = ["has_covid","city"]

In [23]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
).set_output(transform="pandas")

encoded_data = encoder.fit_transform(df[categorical_columns])
display(encoded_data.head())

,has_covid_No,has_covid_Yes,city_Bangalore,city_Delhi,city_Kolkata,city_Mumbai
0,1.0,0.0,0.0,0.0,1.0,0.0
1,0.0,1.0,0.0,1.0,0.0,0.0
2,1.0,0.0,0.0,1.0,0.0,0.0
3,1.0,0.0,0.0,0.0,1.0,0.0
4,1.0,0.0,0.0,0.0,0.0,1.0


In [30]:
x = df.drop("has_covid",axis=1)
y = df["has_covid"]

In [36]:
x_numeric = x.select_dtypes(include=np.number)

# Map categorical target y to numeric values
y_numeric = y.map({'No': 0, 'Yes': 1})

selector = SelectKBest(
    score_func=f_regression,
    k=2
).set_output(transform="pandas")

x_selected = selector.fit_transform(x_numeric, y_numeric)
print(x_selected.head())

ValueError: Input X contains NaN.
SelectKBest does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values